In [1]:
import duckdb
import os

print("DuckDB is ready!")

con = duckdb.connect()

print("Connection established!")

DuckDB is ready!
Connection established!


In [2]:
import duckdb
import os

# Location of the original dataset
csv_path = r"D:\IIT\Year 4\IRP\transactions_train.csv"

# Check whether the file exists
if os.path.exists(csv_path):
    print("Dataset found!")
    print(f"File size: {os.path.getsize(csv_path) / (1024**3):.2f} GB")
else:
    print("Dataset not found. Please check the file path.")

# Establish DuckDB connection
con = duckdb.connect()

print("DuckDB connection established!")

Dataset found!
File size: 3.25 GB
DuckDB connection established!


In [3]:
# Read the CSV using DuckDB
# This does NOT load the entire 3.2 GB file into Pandas.

transactions = con.read_csv(csv_path)

# Display column names
print("DATASET COLUMNS:")
print(transactions.columns)

# Display column data types
print("\nDATA TYPES:")
print(transactions.types)

# Display the first 5 records
print("\nFIRST 5 RECORDS:")
display(transactions.limit(5).df())

DATASET COLUMNS:
['t_dat', 'customer_id', 'article_id', 'price', 'sales_channel_id']

DATA TYPES:
[DATE, VARCHAR, VARCHAR, DOUBLE, BIGINT]

FIRST 5 RECORDS:


,t_dat,customer_id,article_id,price,sales_channel_id
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0663713001,0.050831,2
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0541518023,0.030492,2
2,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,0505221004,0.015237,2
3,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,0685687003,0.016932,2
4,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,0685687004,0.016932,2


In [4]:
record_count = con.execute("""
    SELECT COUNT(*) 
    FROM read_csv_auto(?)
""", [csv_path]).fetchone()[0]

print(f"Total records: {record_count:,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total records: 31,788,324


In [5]:
quality_summary = con.execute("""
    SELECT
        COUNT(*) AS total_records,
        MIN(t_dat) AS first_transaction_date,
        MAX(t_dat) AS last_transaction_date,

        COUNT(*) FILTER (WHERE t_dat IS NULL) AS missing_dates,
        COUNT(*) FILTER (WHERE customer_id IS NULL) AS missing_customers,
        COUNT(*) FILTER (WHERE article_id IS NULL) AS missing_articles,
        COUNT(*) FILTER (WHERE price IS NULL) AS missing_prices,
        COUNT(*) FILTER (WHERE sales_channel_id IS NULL) AS missing_channels,

        COUNT(*) FILTER (WHERE price <= 0) AS non_positive_prices

    FROM read_csv_auto(?)
""", [csv_path]).df()

display(quality_summary)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_records,first_transaction_date,last_transaction_date,missing_dates,missing_customers,missing_articles,missing_prices,missing_channels,non_positive_prices
0,31788324,2018-09-20,2020-09-22,0,0,0,0,0,0


In [6]:
duplicate_summary = con.execute("""
    SELECT
        SUM(record_count - 1) AS duplicate_records
    FROM (
        SELECT COUNT(*) AS record_count
        FROM read_csv_auto(?)
        GROUP BY
            t_dat,
            customer_id,
            article_id,
            price,
            sales_channel_id
        HAVING COUNT(*) > 1
    )
""", [csv_path]).df()

display(duplicate_summary)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,duplicate_records
0,2974905.0
